# Notebook 2 · Agente analista: LLM + ferramentas + RAG + skills

**O caso:** a Aurora Energia, uma comercializadora fictícia, planeja em outubro os contratos de compra de energia
dos 12 meses seguintes. No notebook 1, a turma explorou a carga mensal do Sistema Interligado Nacional (SIN) e comparou
sete modelos de previsão. Agora um **agente analista** usa esses resultados para explicar, simular cenários e escrever
o memorando para a diretoria.

Este notebook tem duas partes:

3. **Agente analista:** um LLM aberto do Hugging Face com ferramentas, RAG e skills.
4. **O LLM como previsor:** o LLM tenta prever os números da série, e comparamos com o Chronos-2.

**Roda sozinho:** não é preciso rodar o notebook 1 antes. Os resultados dele já estão salvos na pasta `resultados/`
do repositório.

Antes de começar: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.

In [ ]:
import os, re, sys, time, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("AULA_SMOKE") == "1"   # teste automatizado em CPU, com um LLM menor
REPO = "https://github.com/pgcferreira-stack/aula-series-llm-fgv"

if IN_COLAB:
    !pip uninstall -q -y torchao
    !pip install -q smolagents sentence-transformers "transformers>=4.51"
    if not os.path.exists("aula-series-llm-fgv"):
        !git clone -q --depth 1 {REPO}
    os.chdir("aula-series-llm-fgv")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("Device:", DEVICE)

# Parte 3 — Agente analista: LLM + ferramentas + RAG + skills

## 3.0 Os dados e os resultados do notebook 1

O agente não refaz as previsões. Ele lê a série do ONS e os resultados do backtest salvos pelo notebook 1.

In [ ]:
from statsmodels.tsa.seasonal import STL

dados = pd.read_csv("dados/carga_sin_mensal.csv", parse_dates=["mes"]).set_index("mes")
carga = dados["carga_mwmed"]
stl = STL(carga, period=12, robust=True).fit()
comp = pd.DataFrame({"c": carga.pct_change(12), "i": dados["ibc_br"].pct_change(12)}).dropna().loc["2004":"2022"]
comp = comp.drop(pd.date_range("2020-03-01", "2021-06-01", freq="MS"), errors="ignore")
elasticidade = np.polyfit(comp.i, comp.c, 1)[0]
PLD, HORAS_MES, PARTICIPACAO = 250, 730, 0.02   # R$/MWh de referência, horas por mês, fatia da Aurora no SIN

H = 12
ULTIMO = carga.index.max()
ORIGENS = [ULTIMO - pd.DateOffset(months=H * k) for k in (3, 2, 1)]
serie = pd.DataFrame({"unique_id": "SIN", "ds": carga.index, "y": carga.values})
def futuro(origem):
    return pd.date_range(origem + pd.DateOffset(months=1), periods=H, freq="MS")

ranking = pd.read_csv("resultados/ranking.csv")
backtest = pd.read_csv("resultados/backtest.csv", parse_dates=["ds", "origem"])
previsao = pd.read_csv("resultados/previsao_12_meses.csv", parse_dates=["mes"])
print("Resultados do notebook 1:", len(ranking), "modelos,", len(previsao), "meses de previsão")
ranking[["modelo", "MAPE (%)", "Custo anual do erro (R$ mi)"]]

## 3.1 O LLM

In [ ]:
from smolagents import TransformersModel

MODELO_LLM = "Qwen/Qwen3-0.6B" if SMOKE else "Qwen/Qwen3-4B-Instruct-2507"
llm = TransformersModel(model_id=MODELO_LLM, device_map=DEVICE,
                        torch_dtype="float16" if DEVICE == "cuda" else "float32",
                        max_new_tokens=128 if SMOKE else 1024,
                        apply_chat_template_kwargs={"enable_thinking": False})

def perguntar(texto):
    """Uma pergunta direta ao LLM, sem ferramentas."""
    return llm.generate([{"role": "user", "content": [{"type": "text", "text": texto}]}]).content

print(perguntar("Em uma frase: o que é a carga de energia do Sistema Interligado Nacional?"))

## 3.2 RAG: a base de documentos

A pasta `documentos/` tem textos sobre os eventos que marcaram a série, a política de contratação da Aurora Energia e um glossário.
Cada parágrafo vira um embedding; a busca devolve os parágrafos mais parecidos com a pergunta.

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer

trechos = []
for arquivo in sorted(Path("documentos").glob("*.md")):
    texto = arquivo.read_text()
    titulo = texto.splitlines()[0].lstrip("# ").strip()
    for paragrafo in texto.split("\n\n")[1:]:
        if len(paragrafo.strip()) > 40:
            trechos.append({"fonte": titulo, "texto": paragrafo.strip()})

embedder = SentenceTransformer("intfloat/multilingual-e5-small", device=DEVICE)
E_trechos = embedder.encode([f"passage: {t['fonte']}. {t['texto']}" for t in trechos], normalize_embeddings=True)

def buscar(consulta, k=3):
    e = embedder.encode([f"query: {consulta}"], normalize_embeddings=True)[0]
    ordem = (E_trechos @ e).argsort()[::-1][:k]
    return [trechos[i] | {"similaridade": float(E_trechos[i] @ e)} for i in ordem]

print(len(trechos), "trechos indexados")
for t in buscar("por que a carga caiu em 2020?"):
    print(f"{t['similaridade']:.2f} [{t['fonte']}] {t['texto'][:100]}...")

## 3.3 Skills

Uma *skill* é um procedimento escrito em linguagem natural, guardado em um arquivo `SKILL.md`, com nome e descrição no cabeçalho.
O agente vê só as descrições; quando uma tarefa pede uma skill, ele carrega as instruções completas.

In [ ]:
import re

def ler_skill(caminho):
    texto = caminho.read_text()
    cabecalho = dict(re.findall(r"^(name|description):\s*(.+)$", texto.split("---")[1], re.M))
    return cabecalho["name"], cabecalho["description"], texto.split("---", 2)[2].strip()

SKILLS = {nome: (desc, corpo) for nome, desc, corpo in map(ler_skill, sorted(Path("skills").glob("*/SKILL.md")))}
for nome, (desc, _) in SKILLS.items():
    print(f"- {nome}: {desc}")
print("\n" + Path("skills/relatorio_executivo/SKILL.md").read_text())

## 3.4 As ferramentas do agente

In [ ]:
from smolagents import tool

@tool
def ranking_modelos() -> str:
    """Devolve a tabela de erro de cada modelo no backtest de 12 meses, do melhor para o pior."""
    colunas = ["modelo", "MAPE (%)", "MAE (MWmed)", "Viés (MWmed)", "Custo anual do erro (R$ mi)"]
    return ranking[colunas].round(2).to_string(index=False)

@tool
def previsao_12_meses() -> str:
    """Devolve a previsão de carga (MWmed) dos próximos 12 meses, mês a mês, com o intervalo de 80% do Chronos-2."""
    return previsao.assign(mes=previsao.mes.dt.strftime("%m/%Y")).round(0).to_string(index=False)

@tool
def maiores_desvios(n: int) -> str:
    """Lista os meses em que a carga mais se afastou do padrão de tendência e sazonalidade (resíduo da decomposição STL).

    Args:
        n: quantos meses listar
    """
    r = (stl.resid / carga * 100).round(1)
    top = r.abs().nlargest(int(n)).index
    return "\n".join(f"{m:%m/%Y}: {r[m]:+.1f}% em relação ao esperado" for m in sorted(top))

@tool
def cenario_atividade(variacao_pct: float) -> str:
    """Simula o efeito sobre a carga dos próximos 12 meses de uma variação adicional na atividade econômica (IBC-Br).

    Args:
        variacao_pct: variação adicional do IBC-Br em pontos percentuais, por exemplo 2 ou -1.5
    """
    base = previsao["Chronos-2"].mean()
    efeito = elasticidade * float(variacao_pct) / 100
    return (f"Carga média prevista (base): {base:,.0f} MWmed. Elasticidade histórica: {elasticidade:.2f}. "
            f"Cenário com {variacao_pct:+.1f} p.p. de IBC-Br: {base * (1 + efeito):,.0f} MWmed "
            f"({efeito * 100:+.2f}%, {base * efeito:+,.0f} MWmed).")

@tool
def custo_do_erro(erro_mwmed: float, meses: int) -> str:
    """Calcula o custo financeiro para a Aurora Energia de um erro na previsão da carga do SIN.

    Args:
        erro_mwmed: tamanho do erro na carga do SIN, em MWmed
        meses: por quantos meses o erro dura
    """
    erro_carteira = abs(float(erro_mwmed)) * PARTICIPACAO
    custo = erro_carteira * HORAS_MES * PLD * int(meses)
    return (f"Um erro de {abs(float(erro_mwmed)):,.0f} MWmed na carga do SIN equivale a {erro_carteira:,.0f} MWmed na carteira "
            f"da Aurora (2% do SIN). Por {int(meses)} meses, custa cerca de R$ {custo / 1e6:,.1f} milhões (PLD de R$ {PLD}/MWh).")

@tool
def buscar_documentos(consulta: str) -> str:
    """Busca nos documentos da empresa (eventos históricos, política de contratação, glossário) os trechos mais relevantes.

    Args:
        consulta: o que procurar, em linguagem natural
    """
    return "\n\n".join(f"[{t['fonte']}] {t['texto']}" for t in buscar(consulta))

@tool
def carregar_skill(nome: str) -> str:
    """Carrega as instruções completas de uma skill. Use antes de executar uma tarefa que corresponda a uma skill.

    Args:
        nome: o nome da skill, por exemplo relatorio_executivo
    """
    return SKILLS[nome][1] if nome in SKILLS else f"Skill '{nome}' não existe. Disponíveis: {list(SKILLS)}"

FERRAMENTAS = [ranking_modelos, previsao_12_meses, maiores_desvios, cenario_atividade,
               custo_do_erro, buscar_documentos, carregar_skill]
print(custo_do_erro(1000, 12))

## 3.5 O agente

In [ ]:
from smolagents import ToolCallingAgent

INSTRUCOES = ("Você é o analista de planejamento da Aurora Energia. Responda em português, de forma objetiva, "
              "usando apenas números devolvidos pelas ferramentas.\n"
              "Skills disponíveis (carregue com carregar_skill antes de usar):\n"
              + "\n".join(f"- {nome}: {desc}" for nome, (desc, _) in SKILLS.items()))

agente = ToolCallingAgent(tools=FERRAMENTAS, model=llm, instructions=INSTRUCOES,
                          max_steps=3 if SMOKE else 8, verbosity_level=1)

def perguntar_ao_agente(pergunta):
    try:
        return agente.run(pergunta)
    except Exception as erro:
        return f"O agente não concluiu: {erro}"

resposta = perguntar_ao_agente("Qual modelo teve o menor erro no backtest e qual foi o MAPE?")
print("\nRESPOSTA:", resposta)

In [ ]:
resposta = perguntar_ao_agente("Por que a carga caiu tanto em abril de 2020? Isso deve afetar a previsão?")
print("\nRESPOSTA:", resposta)

In [ ]:
resposta = perguntar_ao_agente("E se a atividade econômica crescer 2 pontos percentuais acima do esperado? Quanto isso custa se não ajustarmos os contratos?")
print("\nRESPOSTA:", resposta)

In [ ]:
memorando = perguntar_ao_agente("Escreva o memorando de recomendação de contratação para a diretoria.")
print("\nMEMORANDO:\n", memorando)

# Parte 4 — O LLM como previsor

## 4.1 O LLM prevê os números?

Um LLM também pode receber a série escrita como texto e "continuar" os números. Vamos comparar com o Chronos-2 no último ponto de corte.

In [ ]:
origem = ORIGENS[-1]
hist = serie[serie.ds <= origem].tail(48)
texto_serie = ", ".join(f"{v / 1000:.1f}" for v in hist.y)
pedido = (f"Esta é a carga mensal de energia do Brasil, em mil MWmed, nos últimos 48 meses, do mais antigo para o mais recente:\n"
          f"{texto_serie}\n\nPreveja os próximos 12 meses. Responda SOMENTE com 12 números separados por vírgula, sem texto.")
bruto = perguntar(pedido)
numeros = [float(x) * 1000 for x in re.findall(r"\d+(?:\.\d+)?", bruto)][:H]
real = serie[serie.ds.isin(futuro(origem))].y.values
chr_ult = backtest[backtest.origem == origem]["Chronos-2"].values

if len(numeros) == H:
    mape_llm = np.mean(np.abs(np.array(numeros) - real) / real) * 100
    print(f"MAPE do LLM como previsor: {mape_llm:.1f}% | Chronos-2: {np.mean(np.abs(chr_ult - real) / real) * 100:.1f}%")
    fig, eixo = plt.subplots(figsize=(12, 4))
    eixo.plot(hist.ds, hist.y / 1000, color="gray")
    eixo.plot(futuro(origem), real / 1000, color="black", lw=2, label="realizado")
    eixo.plot(futuro(origem), np.array(numeros) / 1000, label=f"LLM ({MODELO_LLM.split('/')[-1]})")
    eixo.plot(futuro(origem), chr_ult / 1000, label="Chronos-2")
    eixo.legend(); eixo.set_ylabel("mil MWmed"); plt.show()
else:
    print(f"O LLM devolveu {len(numeros)} números em vez de {H}:\n{bruto[:300]}")

## Resumo

| Etapa | O que foi feito |
|---|---|
| Agente | LLM aberto com 7 ferramentas, RAG sobre documentos e skills |
| Memorando | O agente escreve a recomendação de contratação para a diretoria |
| LLM como previsor | O LLM prevê os números da série e é comparado com o Chronos-2 |